# RAGBench EDA — `01` (Group 16)

Confirms the **real dataset schema** and answers the **chunking-lever question** *before* we build the TRACe evaluator. No GPU, no LLM calls — pure data inspection. Run top-to-bottom (**Runtime → Run all**); first run downloads + caches (a few minutes).

**What it answers**
1. Real config names + how they map to the 5 domains (vs our assumptions).
2. Exact field **names & types** — `AI_CONTEXT.md §8` is currently *unverified*.
3. **Structure** of the sentence-keyed + support-annotation fields (the evaluator is built on these).
4. Reference **TRACe score** fields + their value ranges.
5. Per sub-dataset: are docs **pre-segmented** or do they need **re-chunking**? (`§22` open question).

➡️ **After running, paste back to Claude** the outputs of the cells labelled **(1) configs**, **(2) features**, **(2b) field-by-field**, **(3) sentence-field structure**, **(4) reference scores**, and the two **summary tables** ((5b) and (6)). Then we lock the truth into `AI_CONTEXT.md §8` and build the evaluator.

In [ ]:
# One-time install (Colab). `datasets` pulls in pyarrow + huggingface_hub.
!pip install -q datasets

In [ ]:
import json, statistics, textwrap, os
import pandas as pd
from datasets import get_dataset_config_names, load_dataset, load_dataset_builder

REPO = "rungalileo/ragbench"
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)
print("datasets repo:", REPO)

In [ ]:
# OUR ASSUMPTIONS (from AI_CONTEXT.md §8) — NOT trusted. The next cells print reality and diff against this.
EXPECTED_DOMAIN_MAP = {
    "pubmedqa":   "Biomedical Research",
    "covidqa":    "Biomedical Research",
    "hotpotqa":   "General Knowledge",
    "msmarco":    "General Knowledge",
    "hagrid":     "General Knowledge",
    "expertqa":   "General Knowledge",
    "cuad":       "Legal",
    "techqa":     "Customer Support",
    "emanual":    "Customer Support",
    "delucionqa": "Customer Support",
    "finqa":      "Finance",
    "tatqa":      "Finance",
}
print(len(EXPECTED_DOMAIN_MAP), "expected sub-datasets")

In [ ]:
# (1) CONFIGS — confirm the REAL config names and their domain mapping.
configs = get_dataset_config_names(REPO)
print(f"{len(configs)} configs found on HuggingFace:\n")
for c in sorted(configs):
    print(f"  {c:14s} -> {EXPECTED_DOMAIN_MAP.get(c, '!!! NOT IN OUR EXPECTED MAP')}")

missing = sorted(set(EXPECTED_DOMAIN_MAP) - set(configs))
extra   = sorted(set(configs) - set(EXPECTED_DOMAIN_MAP))
print("\nExpected but MISSING from real configs:", missing or "none")
print("Real configs we did NOT expect:        ", extra or "none")

In [ ]:
# (2) FEATURES — deep-dive schema on one small/short-doc config to confirm exact field names & types.
PREFERRED_PROBES = ["pubmedqa", "covidqa", "delucionqa", "hotpotqa"]
PROBE = next((c for c in PREFERRED_PROBES if c in configs), configs[0])
print("Probe config:", PROBE, "\n")

dd = load_dataset(REPO, PROBE)
print("Splits & sizes:", {k: len(v) for k, v in dd.items()}, "\n")

feats = dd["test"].features
print("=== FEATURES (test split) ===")
for name, ftype in feats.items():
    print(f"  {name:34s} : {ftype}")

In [ ]:
# (2b) FIELD-BY-FIELD — dump ONE full example (values truncated).
ex = dd["test"][0]
print(f"{PROBE} test[0] — {len(ex)} fields\n")
for k, v in ex.items():
    vt = type(v).__name__
    if isinstance(v, (list, tuple)):
        head = repr(v[0])[:280] if v else "(empty)"
        preview = f"list(len={len(v)}); first elem: {head}"
    elif isinstance(v, dict):
        preview = f"dict(keys={list(v.keys())})"
    else:
        preview = repr(v)[:280]
    print(f"--- {k}  ({vt})")
    print("    " + preview.replace("\n", "\n    "))

In [ ]:
# (3) EVALUATOR FIELDS — presence of the fields TRACe depends on + STRUCTURE of the keyed-sentence fields.
CANDIDATE_FIELDS = [
    "id", "question", "documents", "documents_sentences",
    "response", "response_sentences",
    "all_relevant_sentence_keys", "all_utilized_sentence_keys",
    "sentence_support_information", "unsupported_response_sentence_keys",
    "overall_supported", "adherence_score", "relevance_score",
    "utilization_score", "completeness_score", "dataset_name",
]
print("Expected/candidate field presence in", PROBE, ":")
for f in CANDIDATE_FIELDS:
    print(f"  [{'x' if f in ex else ' '}] {f}")
print("\nActual fields NOT in our candidate list:",
      [k for k in ex if k not in CANDIDATE_FIELDS] or "none")

for f in ["documents_sentences", "response_sentences", "sentence_support_information"]:
    if f in ex:
        val = ex[f]
        n = len(val) if hasattr(val, "__len__") else "n/a"
        print(f"\n=== STRUCTURE of '{f}' ===  type={type(val).__name__}  len={n}")
        sample = val[:2] if isinstance(val, (list, tuple)) else val
        print("first element(s) (repr, truncated):")
        print(textwrap.indent(repr(sample)[:1400], "  "))

In [ ]:
# (4) REFERENCE SCORES — confirm names, types, and value ranges of the ground-truth TRACe labels.
SCORE_GUESSES = ["relevance_score", "utilization_score", "completeness_score",
                 "adherence_score", "overall_supported"]
present = [s for s in SCORE_GUESSES if s in ex]
print("Reference-score fields present (by our guessed names):", present or "NONE")

print("\nAny field whose name hints at a score/label:")
for k in ex:
    kl = k.lower()
    if any(t in kl for t in ["score", "support", "relevan", "util", "complet", "adher"]):
        print(f"   {k:34s} = {repr(ex[k])[:120]}")

samp = dd["test"].select(range(min(300, len(dd["test"]))))
print("\nRanges over a", len(samp), "row sample:")
for s in present:
    col = samp[s]
    nums = [x for x in col if isinstance(x, (int, float)) and not isinstance(x, bool)]
    if nums:
        print(f"  {s:24s} n={len(nums):4d}  min={min(nums):.3f}  max={max(nums):.3f}  mean={sum(nums)/len(nums):.3f}")
    else:
        print(f"  {s:24s} non-numeric -> sample {col[:5]}")

In [ ]:
# (5) Helpers for the pre-segmentation / chunking-lever analysis (AI_CONTEXT §22).
def words(t):
    return len(t.split()) if isinstance(t, str) else 0

def find_docs_field(row):
    if row.get("documents") is not None:
        return "documents"
    for k, v in row.items():           # fallback: first list-of-strings field
        if isinstance(v, list) and v and all(isinstance(x, str) for x in v):
            return k
    return None

def pct(xs, p):
    if not xs:
        return None
    s = sorted(xs)
    return s[min(len(s) - 1, int(round(p * (len(s) - 1))))]

def doc_stats_for(config, n=200):
    try:
        ds = load_dataset(REPO, config, split="test")
    except Exception as e:
        return {"config": config, "error": str(e)[:140]}
    ds = ds.select(range(min(n, len(ds))))
    field, docs_per_ex, wlens = None, [], []
    for row in ds:
        if field is None:
            field = find_docs_field(row)
        d = row.get(field) if field else None
        if d is None:
            continue
        if isinstance(d, str):
            d = [d]
        docs_per_ex.append(len(d))
        wlens.extend(words(s) for s in d)
    med_w, p95_w = pct(wlens, 0.5), pct(wlens, 0.95)
    med_d = statistics.median(docs_per_ex) if docs_per_ex else None
    if med_w is None:
        lever = "?"
    elif (p95_w or 0) > 800:
        lever = "HIGH (long docs -> re-chunk)"
    elif med_w < 200 and (med_d or 0) >= 3:
        lever = "LOW (pre-segmented short passages)"
    else:
        lever = "MEDIUM"
    return {"config": config, "domain": EXPECTED_DOMAIN_MAP.get(config, "?"),
            "docs_field": field, "n": len(ds),
            "docs/ex_med": med_d, "docs/ex_max": max(docs_per_ex, default=None),
            "word_len_med": med_w, "word_len_p95": p95_w,
            "word_len_max": max(wlens, default=None), "chunking_lever": lever}

In [ ]:
# (5b) SUMMARY TABLE — cross-config document stats (samples test split of each config).
rows = []
for c in sorted(configs):
    print("scanning", c, "...")
    rows.append(doc_stats_for(c, n=200))
summary = pd.DataFrame(rows)
summary

In [ ]:
# (6) SPLIT SIZES per config (cheap via builder metadata; falls back to a test-split load).
size_rows = []
for c in sorted(configs):
    row = {"config": c, "domain": EXPECTED_DOMAIN_MAP.get(c, "?")}
    try:
        spl = (load_dataset_builder(REPO, c).info.splits) or {}
        if spl:
            for k, v in spl.items():
                row[k] = v.num_examples
        else:
            row["test"] = len(load_dataset(REPO, c, split="test"))
            row["note"] = "test-load (no builder split info)"
    except Exception as e:
        row["error"] = str(e)[:120]
    size_rows.append(row)
sizes = pd.DataFrame(size_rows)
sizes

In [ ]:
# (optional) Visualise the contrast: shortest- vs longest-doc config (words per document).
try:
    import matplotlib.pyplot as plt
    valid = summary.dropna(subset=["word_len_p95"])
    short_cfg = valid.loc[valid["word_len_p95"].idxmin(), "config"]
    long_cfg  = valid.loc[valid["word_len_p95"].idxmax(), "config"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
    for ax, cfg in zip(axes, [short_cfg, long_cfg]):
        full = load_dataset(REPO, cfg, split="test")
        ds = full.select(range(min(200, len(full))))
        fld = find_docs_field(ds[0])
        wl = []
        for r in ds:
            d = r.get(fld)
            d = [d] if isinstance(d, str) else (d or [])
            wl.extend(words(s) for s in d)
        ax.hist(wl, bins=40)
        ax.set_title(f"{cfg}  (median {pct(wl, 0.5)} words/doc)")
        ax.set_xlabel("words per document"); ax.set_ylabel("count")
    plt.tight_layout(); plt.show()
    print("short-doc config:", short_cfg, "| long-doc config:", long_cfg)
except Exception as e:
    print("plot skipped:", e)

In [ ]:
# (7) SAVE summaries (download from the Colab Files panel, or commit into the repo's results/).
summary.to_csv("ragbench_doc_summary.csv", index=False)
sizes.to_csv("ragbench_split_sizes.csv", index=False)
schema_dump = {
    "probe_config": PROBE,
    "all_configs": sorted(configs),
    "splits": {k: len(v) for k, v in dd.items()},
    "fields": {name: str(ftype) for name, ftype in feats.items()},
    "example_field_types": {k: type(v).__name__ for k, v in ex.items()},
}
with open("ragbench_schema.json", "w") as f:
    json.dump(schema_dump, f, indent=2)
print("Wrote: ragbench_doc_summary.csv, ragbench_split_sizes.csv, ragbench_schema.json")

## What to paste back to Claude

So we can lock the schema into `AI_CONTEXT.md §8` and build the evaluator, copy back the **printed output** of:

- **(1) configs** — real names + the MISSING / unexpected diff lines
- **(2) features** + **(2b) field-by-field** — the exact field names & types
- **(3)** — the `documents_sentences` / `response_sentences` / `sentence_support_information` **structure** dumps
- **(4)** — which reference-score fields exist and their ranges
- **(5b)** and **(6)** — the two summary tables (doc stats + split sizes)

Trimming long values is fine — the **field names, types, and the keyed-sentence structure** are what matter most.